We will integrate two projects that are project 13 and project here. Project 13 contains a context-aware chatbot i.e, if you ask question like name, it will answer you the name if you ask weather it will ask you the question which city and then move on, and project 14 contains an intent classifier i.e, when you gave input like todays temprature is 25 degreee  it will gave you output of weather query since we are talking about query, and now lets integrate both of them for more better results.

In [1]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import re
import sqlite3
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import spacy

from transformers import pipeline

import speech_recognition as sr
import pyttsx3

print("Libraries imported successfully.")

c:\MiniForge\envs\intern_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully.


In [2]:
# ============================================================
# CELL 2: LOAD TRAINED INTENT CLASSIFIER AND TRANSFORMER
# ============================================================

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Model folder path
MODEL_DIR = "models"

# Load trained SVM intent classifier from Project 14
intent_model = joblib.load(os.path.join(MODEL_DIR, "best_model.pkl"))

# Load TF-IDF vectorizer
tfidf_vectorizer = joblib.load(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"))

# Load label encoder
label_encoder = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))

# Load transformer fallback model from Project 13
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("SVM intent classifier and transformer fallback loaded successfully.")

c:\MiniForge\envs\intern_env\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


SVM intent classifier and transformer fallback loaded successfully.


In [3]:
# ============================================================
# CELL 3: RESPONSE SYSTEM
# ============================================================

responses = {
    "greeting": "Hello! How can I help you today?",
    "goodbye": "Goodbye! Have a great day.",
    "thanks": "You're welcome. Let me know if you need anything else.",

    "food_order": "Sure, I can help you with food ordering. What would you like to order?",
    "weather_query": "Please tell me your city so I can help with the weather information.",
    "name_query": "I am your AI assistant, designed to understand your intent and help you.",

    "account_help": "Sure, I can help with your account. Please tell me what issue you are facing.",
    "payment_issue": "I can help with payment issues. Please explain the payment problem.",
    "order_status": "I can help you track your order. Please provide your order ID or tracking number.",
    "appointment_booking": "Sure, I can help you book an appointment. Please tell me your preferred date and time.",
    "product_search": "Sure, I can help you find a product. What product are you looking for?",
    "complaint": "I'm sorry to hear that. Please explain the issue so I can help you properly.",

    "password_reset": "You can reset your password using the 'Forgot Password' option. I can guide you step by step.",
    "business_hours": "Our support team is available from 9 AM to 6 PM, Monday to Saturday.",
    "cancellation": "I can help you cancel your subscription. Please confirm if you want to continue.",
    "return_request": "I can help you start a return request. Please share your order number.",
    "service_info": "We provide account support, order tracking, returns, payment help, cancellation support, and technical assistance.",
    "technical_support": "I can help with technical support. Please describe the issue you are facing.",

    "unknown_intent": "Sorry, I could not clearly understand your request. Please explain it again.",
    "empty_input": "Please type your message."
}

candidate_labels = list(responses.keys())

print("Response system loaded successfully.")

Response system loaded successfully.


In [4]:
# ============================================================
# CELL 4: TEXT PREPROCESSING
# ============================================================

def preprocess_text(text):
    """
    Clean user input using the same preprocessing used in Project 14.
    """

    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\$\s?\d+(\.\d+)?", " money ", text)
    text = re.sub(r"\d+(\.\d+)?", " number ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    doc = nlp(text)

    cleaned_words = []

    for token in doc:
        if (
            not token.is_stop
            and not token.is_punct
            and not token.is_space
            and len(token.text) > 2
        ):
            cleaned_words.append(token.lemma_)

    return " ".join(cleaned_words)

In [5]:
# ============================================================
# CELL 5: RULE-BASED INTENT DETECTION
# ============================================================

def rule_based_intent(user_input):
    text = user_input.lower().strip()

    greetings = ["hi", "hello", "hey", "salam", "assalamualaikum"]
    thanks = ["ok", "okay", "thanks", "thank you", "alright", "fine"]
    goodbye = ["bye", "goodbye", "see you", "see you later"]

    weather_words = [
        "weather", "wether", "temperature", "temprature", "temp",
        "hot", "cold", "rain", "raining", "sunny", "cloudy",
        "forecast", "humidity", "wind", "degree", "degrees"
    ]

    food_words = [
        "pizza", "burger", "biryani", "food", "meal",
        "drink", "sandwich", "fries", "chicken"
    ]

    if text in greetings:
        return "greeting", 1.0

    if text in thanks:
        return "thanks", 1.0

    if text in goodbye:
        return "goodbye", 1.0

    if any(word in text for word in weather_words):
        return "weather_query", 1.0

    if any(word in text for word in food_words) or "order food" in text:
        return "food_order", 1.0

    if any(word in text for word in ["cancel", "cancellation", "unsubscribe"]):
        return "cancellation", 1.0

    if any(word in text for word in ["forgot password", "forget password", "reset password", "password"]):
        return "password_reset", 1.0

    if any(word in text for word in ["package", "parcel", "tracking", "shipment", "order id", "order status"]):
        return "order_status", 1.0

    if text.startswith("where is my order") or text.startswith("track my order"):
        return "order_status", 1.0

    if any(word in text for word in ["return", "refund", "replace"]):
        return "return_request", 1.0

    if any(word in text for word in ["payment", "billing", "card", "pay"]):
        return "payment_issue", 1.0

    if any(word in text for word in ["technical", "support", "error", "issue", "problem", "not working"]):
        return "technical_support", 1.0

    if any(word in text for word in ["service", "services", "offer", "provide"]):
        return "service_info", 1.0

    if any(word in text for word in ["open", "hours", "working hours"]):
        return "business_hours", 1.0

    return None, 0.0

In [6]:
# ============================================================
# CELL 6: SVM INTENT PREDICTION
# ============================================================

def predict_intent_svm(user_input, threshold=0.60):
    """
    Predict intent using trained SVM model from Project 14.
    """

    clean_input = preprocess_text(user_input)

    input_vector = tfidf_vectorizer.transform([clean_input])

    predicted_label = intent_model.predict(input_vector)[0]

    intent = label_encoder.inverse_transform([predicted_label])[0]

    probabilities = intent_model.predict_proba(input_vector)[0]

    confidence = float(np.max(probabilities))

    if confidence < threshold:
        return None, confidence

    return intent, confidence

In [7]:
# ============================================================
# CELL 7: TRANSFORMER FALLBACK INTENT PREDICTION
# ============================================================

def predict_intent_transformer(user_input, threshold=0.40):
    """
    Use transformer fallback only if confidence is good enough.
    """

    result = zero_shot_classifier(
        user_input,
        candidate_labels=candidate_labels
    )

    intent = result["labels"][0]
    confidence = float(result["scores"][0])

    # reject weak predictions
    if confidence < threshold:
        return "unknown_intent", confidence

    return intent, confidence

In [8]:
# ============================================================
# CELL 8: HYBRID INTENT DETECTION
# ============================================================

def detect_intent(user_input):
    """
    Hybrid detection:
    1. Rule-based detection for simple inputs
    2. Transformer as main intent detector
    3. SVM as backup if transformer confidence is weak
    """

    # First check simple rule-based intents
    rule_intent, rule_confidence = rule_based_intent(user_input)

    if rule_intent is not None:
        return rule_intent, rule_confidence * 100, "Rule Based"

    # Use transformer as main model
    transformer_intent, transformer_confidence = predict_intent_transformer(
        user_input,
        threshold=0.45
    )

    if transformer_intent != "unknown_intent":
        return transformer_intent, transformer_confidence * 100, "Transformer Main"

    # Use SVM as backup
    svm_intent, svm_confidence = predict_intent_svm(
        user_input,
        threshold=0.50
    )

    if svm_intent is not None:
        return svm_intent, svm_confidence * 100, "Trained SVM Backup"

    return "unknown_intent", transformer_confidence * 100, "Unknown"

In [9]:
# ============================================================
# CELL 9: CONTEXT MEMORY
# ============================================================

context_memory = {}

def update_context(user_id, intent):
    context_memory[user_id] = {
        "last_intent": intent,
        "updated_at": datetime.now().isoformat()
    }

def get_last_intent(user_id):
    return context_memory.get(user_id, {}).get("last_intent")

def clear_context(user_id):
    if user_id in context_memory:
        del context_memory[user_id]

def is_possible_city(user_input):
    text = user_input.lower().strip()

    blocked_words = [
        "yes", "no", "ok", "okay", "thanks", "thank you",
        "hi", "hello", "bye", "exit", "clear"
    ]

    if text in blocked_words:
        return False

    if re.search(r"\d", text):
        return False

    if len(text.split()) <= 3 and len(text) >= 2:
        return True

    return False

In [10]:
# ============================================================
# CELL 10: SQLITE CHAT HISTORY
# ============================================================

DB_PATH = "chat_history.db"

def init_database():
    """
    Create SQLite database for chat history.
    """

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT,
            user_message TEXT,
            bot_response TEXT,
            predicted_intent TEXT,
            confidence REAL,
            model_used TEXT,
            timestamp TEXT
        )
    """)

    conn.commit()
    conn.close()


def save_chat(user_id, user_message, bot_response, intent, confidence, model_used):
    """
    Save chat conversation.
    """

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO chat_history
        (
            user_id,
            user_message,
            bot_response,
            predicted_intent,
            confidence,
            model_used,
            timestamp
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        user_id,
        user_message,
        bot_response,
        intent,
        float(confidence),
        model_used,
        datetime.now().isoformat()
    ))

    conn.commit()
    conn.close()


def load_chat_history(limit=10):
    """
    Load recent chat history.
    """

    conn = sqlite3.connect(DB_PATH)

    query = """
        SELECT
            user_message,
            bot_response,
            predicted_intent,
            confidence,
            model_used,
            timestamp
        FROM chat_history
        ORDER BY id DESC
        LIMIT ?
    """

    history = pd.read_sql_query(query, conn, params=(limit,))
    conn.close()

    return history


init_database()

print("Database initialized successfully.")

Database initialized successfully.


In [11]:
# ============================================================
# CELL 11: MAIN INTEGRATED CHATBOT ENGINE
# ============================================================

def chatbot_response(user_input, user_id="notebook_user"):
    if user_input is None or not user_input.strip():
        return {
            "response": "Please type your message.",
            "intent": "empty_input",
            "confidence": 0.0,
            "model_used": "Input Validation"
        }

    user_input = user_input.strip()
    last_intent = get_last_intent(user_id)

    if last_intent == "weather_query" and is_possible_city(user_input):
        response = f"Thank you. I will check the weather information for {user_input.title()}."
        intent = "weather_query_followup"
        confidence = 1.0
        model_used = "Context Memory"
        clear_context(user_id)

    elif last_intent == "order_status" and re.search(r"\d+", user_input):
        response = f"Thank you. I will check your order status using this order/tracking ID: {user_input}"
        intent = "order_status_followup"
        confidence = 1.0
        model_used = "Context Memory"
        clear_context(user_id)

    elif last_intent == "return_request" and re.search(r"\d+", user_input):
        response = f"Thank you. I will start the return process for order ID: {user_input}"
        intent = "return_request_followup"
        confidence = 1.0
        model_used = "Context Memory"
        clear_context(user_id)

    elif last_intent == "cancellation" and user_input.lower() in ["yes", "confirm", "sure", "yes confirm"]:
        response = "Your cancellation request has been confirmed and submitted."
        intent = "cancellation_confirmed"
        confidence = 1.0
        model_used = "Context Memory"
        clear_context(user_id)

    elif last_intent == "food_order":
        response = f"Your food order request has been noted: {user_input}"
        intent = "food_order_followup"
        confidence = 1.0
        model_used = "Context Memory"
        clear_context(user_id)

    elif last_intent == "appointment_booking":
        response = f"Your appointment request has been noted for: {user_input}"
        intent = "appointment_booking_followup"
        confidence = 1.0
        model_used = "Context Memory"
        clear_context(user_id)

    else:
        intent, confidence, model_used = detect_intent(user_input)

        response = responses.get(
            intent,
            "I understand your message, but I need a little more detail to help you properly."
        )

        if intent in [
            "order_status",
            "return_request",
            "cancellation",
            "food_order",
            "appointment_booking",
            "weather_query"
        ]:
            update_context(user_id, intent)
        else:
            clear_context(user_id)

    save_chat(
        user_id=user_id,
        user_message=user_input,
        bot_response=response,
        intent=intent,
        confidence=confidence,
        model_used=model_used
    )

    return {
        "response": response,
        "intent": intent,
        "confidence": round(float(confidence), 3),
        "model_used": model_used
    }

In [12]:
# ============================================================
# CELL 12: TEST INTEGRATED CHATBOT
# ============================================================

test_queries = [
    "hi",
    "I want to order pizza",
    "large chicken pizza",
    "Where is my package?",
    "12345",
    "I forgot my password",
    "I want to cancel my subscription",
    "yes",
    "What is the weather today?",
    "I have a payment issue",
    "I want to book an appointment",
    "tomorrow at 5 PM",
    "thank you",
    "bye"
]

for query in test_queries:
    result = chatbot_response(query)

    print("User:", query)
    print("Bot:", result["response"])
    print("Intent:", result["intent"])
    print("Confidence:", result["confidence"])
    print("Model Used:", result["model_used"])
    print("-" * 70)

User: hi
Bot: Hello! How can I help you today?
Intent: greeting
Confidence: 100.0
Model Used: Rule Based
----------------------------------------------------------------------
User: I want to order pizza
Bot: Sure, I can help you with food ordering. What would you like to order?
Intent: food_order
Confidence: 100.0
Model Used: Rule Based
----------------------------------------------------------------------
User: large chicken pizza
Bot: Your food order request has been noted: large chicken pizza
Intent: food_order_followup
Confidence: 1.0
Model Used: Context Memory
----------------------------------------------------------------------
User: Where is my package?
Bot: I can help you track your order. Please provide your order ID or tracking number.
Intent: order_status
Confidence: 100.0
Model Used: Rule Based
----------------------------------------------------------------------
User: 12345
Bot: Thank you. I will check your order status using this order/tracking ID: 12345
Intent: order_

In [13]:
# ============================================================
# CELL 13: VIEW CHAT HISTORY
# ============================================================

history_df = load_chat_history(limit=10)

history_df

,user_message,bot_response,predicted_intent,confidence,model_used,timestamp
0,bye,Goodbye! Have a great day.,goodbye,100.000000,Rule Based,2026-05-19T13:35:00.532745
1,thank you,You're welcome. Let me know if you need anythi...,thanks,100.000000,Rule Based,2026-05-19T13:35:00.524735
2,tomorrow at 5 PM,Your appointment request has been noted for: t...,appointment_booking_followup,1.000000,Context Memory,2026-05-19T13:35:00.516736
3,I want to book an appointment,"Sure, I can help you book an appointment. Plea...",appointment_booking,85.440327,Trained SVM Backup,2026-05-19T13:35:00.508563
4,I have a payment issue,I can help with payment issues. Please explain...,payment_issue,100.000000,Rule Based,2026-05-19T13:34:54.250683
5,What is the weather today?,Please tell me your city so I can help with th...,weather_query,100.000000,Rule Based,2026-05-19T13:34:54.240691
6,yes,Your cancellation request has been confirmed a...,cancellation_confirmed,1.000000,Context Memory,2026-05-19T13:34:54.231683
7,I want to cancel my subscription,I can help you cancel your subscription. Pleas...,cancellation,100.000000,Rule Based,2026-05-19T13:34:54.222686
8,I forgot my password,You can reset your password using the 'Forgot ...,password_reset,100.000000,Rule Based,2026-05-19T13:34:54.212686
9,12345,Thank you. I will check your order status usin...,order_status_followup,1.000000,Context Memory,2026-05-19T13:34:54.203123


In [14]:
# ============================================================
# CELL 14: INTERACTIVE CONSOLE CHATBOT
# ============================================================

def run_console_chatbot():
    """
    Run chatbot inside notebook or terminal.
    """

    print("Hybrid Context-Aware AI Chatbot Started")
    print("Type 'exit' to stop.")
    print("Type 'clear' to clear context.\n")

    user_id = "console_user"

    while True:
        user_input = input("You: ")

        if user_input.lower().strip() == "exit":
            print("Bot: Goodbye! Have a nice day.")
            break

        if user_input.lower().strip() == "clear":
            clear_context(user_id)
            print("Bot: Context cleared.")
            continue

        result = chatbot_response(user_input, user_id=user_id)

        print("Bot:", result["response"])
        print(f"Intent: {result['intent']} | Confidence: {result['confidence']} | Model: {result['model_used']}")
        print()


# Uncomment to run
# run_console_chatbot()

In [15]:
# ============================================================
# CELL 15: VOICE FEATURES
# ============================================================

tts_engine = pyttsx3.init()
tts_engine.setProperty("rate", 170)
tts_engine.setProperty("volume", 1.0)

def listen_to_user():
    """
    Capture voice input from microphone.
    """

    recognizer = sr.Recognizer()

    try:
        with sr.Microphone() as source:
            print("Listening... Please speak.")
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)

        text = recognizer.recognize_google(audio)
        print("Recognized:", text)
        return text

    except sr.WaitTimeoutError:
        print("No speech detected.")
        return None

    except sr.UnknownValueError:
        print("Could not understand audio.")
        return None

    except sr.RequestError:
        print("Speech recognition service error.")
        return None


def speak_response(text):
    """
    Convert text response into speech.
    """

    try:
        tts_engine.say(text)
        tts_engine.runAndWait()

    except Exception as e:
        print("Text-to-speech error:", e)

In [16]:
# ============================================================
# CELL 16: VOICE ENABLED CHATBOT
# ============================================================

def run_voice_enabled_chat():
    """
    Run chatbot with text and voice support.
    """

    print("Voice-Enabled Hybrid AI Chatbot")
    print("Type 'voice' to use voice input.")
    print("Type 'exit' to stop.")
    print("Type 'clear' to clear context.\n")

    user_id = "voice_console_user"
    voice_mode = False

    while True:
        if voice_mode:
            user_input = listen_to_user()
            voice_mode = False
        else:
            user_input = input("You: ").strip()

        if user_input is None:
            print("Bot: No input detected. Please try again.")
            continue

        if user_input.lower() == "exit":
            print("Bot: Thank you for using the chatbot. Goodbye!")
            break

        if user_input.lower() == "clear":
            clear_context(user_id)
            print("Bot: Conversation context cleared.")
            continue

        if user_input.lower() == "voice":
            voice_mode = True
            print("Switching to voice mode...")
            continue

        result = chatbot_response(user_input, user_id=user_id)

        print("Bot:", result["response"])
        print(f"Intent: {result['intent']} | Confidence: {result['confidence']} | Model: {result['model_used']}")

        # Uncomment if you want spoken responses
        # speak_response(result["response"])


# Uncomment to run
# run_voice_enabled_chat()

In [17]:
# ============================================================
# CELL 17: FINAL PROJECT SUMMARY
# ============================================================

print("""
Hybrid Context-Aware AI Chatbot Completed Successfully!

Final Architecture:
1. Rule-based intent detection for simple inputs
2. Trained SVM intent classifier from Project 14
3. Transformer fallback from Project 13
4. Context memory for follow-up handling
5. SQLite database for chat history
6. Console chatbot interface
7. Voice input and text-to-speech support
""")


Hybrid Context-Aware AI Chatbot Completed Successfully!

Final Architecture:
1. Rule-based intent detection for simple inputs
2. Trained SVM intent classifier from Project 14
3. Transformer fallback from Project 13
4. Context memory for follow-up handling
5. SQLite database for chat history
6. Console chatbot interface
7. Voice input and text-to-speech support

